# 01 · Load Snowflake Metadata into the Neo4j Graph

Run this **after** `00_prerequisites_and_connectivity.ipynb` shows ✅ everywhere.

This notebook builds the semantic layer. It:

1. Loads your configuration and checks what LLM/embedding setup you chose
2. Previews the ingestion (a dry run that touches nothing)
3. Ingests schema metadata from Snowflake into Neo4j (`Database → Schema → Table → Column → Value`)
4. Verifies what landed in the graph (counts, relationships, indexes)
5. Runs a semantic search over the new graph

> ⚠️ **Section 4 writes to Neo4j** (database `NEO4J_DATABASE`). It only *reads* from Snowflake. Re-running is safe: ingestion is idempotent (it updates rather than duplicates).

## 1 · Configuration

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

from dotenv import load_dotenv

env_file = next((p for p in (Path(".env"),) if p.exists()), None)
assert env_file, "No .env found: copy .env.example to .env and fill it in (see docs/attendee_guide.md)"
load_dotenv(env_file, override=True)
print("Config file:", env_file)

PLACEHOLDERS = ("xxxxxxxx", "your-", "xy12345", "workshop_user")
required = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER",
            "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_DATABASE", "SNOWFLAKE_SCHEMA"]
bad = [k for k in required if not os.getenv(k) or any(p in os.getenv(k) for p in PLACEHOLDERS)]
assert not bad, f"Missing or placeholder values in {env_file}: {bad}. Run notebook 0a first."

NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
SF_DATABASE, SF_SCHEMA = os.environ["SNOWFLAKE_DATABASE"], os.environ["SNOWFLAKE_SCHEMA"]
print(f"Source : Snowflake {SF_DATABASE}.{SF_SCHEMA}")
print(f"Target : Neo4j {os.environ['NEO4J_URI']} (database '{NEO4J_DATABASE}')")

### Options you can change

| Setting | Meaning |
|---|---|
| `INGEST_EMBEDDINGS` | `True` embeds table/schema descriptions so semantic (vector) search works. Needs a working `EMBEDDING_MODEL` (section 2 checks it). |
| `VALUE_SAMPLE_LIMIT` | Distinct sample values stored per column. Set `0` if your role has no `SELECT` on the tables (metadata-only). |

In [ ]:
INGEST_EMBEDDINGS = True
VALUE_SAMPLE_LIMIT = int(os.getenv("SNOWFLAKE_VALUE_SAMPLE_LIMIT", "10"))

## 2 · Embedding model check

Embeddings go through LiteLLM, so `EMBEDDING_MODEL` (and its key or `OLLAMA_API_BASE`) in `.env` decides the provider. We send one tiny test request now so a bad key fails here rather than halfway through the ingest.
If you use Ollama, the container from `docs/attendee_guide.md` ("LLM Execution Environments & Fallbacks") must be running.

In [ ]:
import litellm

embedding_model = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
embedding_ok = False
if INGEST_EMBEDDINGS:
    try:
        vec = litellm.embedding(model=embedding_model, input=["connectivity test"]).data[0]["embedding"]
        embedding_ok = True
        print(f"✅ {embedding_model} works (vector dimension {len(vec)})")
    except Exception as e:
        print(f"❌ {embedding_model} failed: {type(e).__name__}: {str(e)[:300]}")
        print("   Fix the key / OLLAMA_API_BASE in .env and re-run this cell, or set INGEST_EMBEDDINGS = False above to continue without semantic search.")
else:
    print("ℹ️  Embeddings disabled: the graph will support catalog and full-text search only.")

USE_EMBEDDINGS = INGEST_EMBEDDINGS and embedding_ok
print("Will ingest with embeddings:", USE_EMBEDDINGS)

## 3 · Dry run (nothing is touched)

Prints the ingestion plan and the settings neocarta resolved. We call the same `neocarta` CLI you would use in a terminal. `neocarta` itself does not read this repo's `.env`, so the notebook passes the values through the environment.

In [ ]:
import re

NEOCARTA = str(Path(sys.executable).parent / "neocarta")
ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
SPINNER = re.compile(r"^[⠀-⣿]\s")   # braille spinner frames

def run_neocarta(*args, show=True):
    # Run the neocarta CLI with the loaded .env values; return (rc, cleaned output)
    env = {**os.environ, "NO_COLOR": "1", "TERM": "dumb"}   # plain text, no spinner animation
    proc = subprocess.run([NEOCARTA, *args], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = [ANSI.sub("", l).rstrip() for l in proc.stdout.splitlines()]
    lines = [l for l in lines if l.strip() and not SPINNER.match(l)]
    text = "\n".join(lines)
    if show:
        print(text)
    return proc.returncode, text

rc, _ = run_neocarta("snowflake", "schema", "--dry-run")
print("\n✅ Dry run OK" if rc == 0 else f"\n❌ Dry run failed (exit {rc}): fix the message above, then re-run")

## 4 · Ingest metadata into Neo4j ✍️

Reads `INFORMATION_SCHEMA` (tables, columns, keys) and sampled values from Snowflake, writes them to Neo4j, then (optionally) embeds descriptions. This takes a minute or two.

In [ ]:
args = ["snowflake", "schema", "--value-sample-limit", str(VALUE_SAMPLE_LIMIT)]
if USE_EMBEDDINGS:
    args += ["--embeddings"]
rc, out = run_neocarta(*args)
print("\n✅ Ingest succeeded" if rc == 0 else f"\n❌ Ingest failed (exit {rc}): see the message above")
assert rc == 0

## 5 · Verify the graph

What we expect for a schema: one `Database`, one `Schema`, a `Table` per table, a `Column` per column, and `Value` nodes for sampled values (when sampling is on).

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(os.environ["NEO4J_URI"],
                              auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))

def q(cypher, **params):
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.run(cypher, **params).data()

print("Nodes:")
for r in q("MATCH (n) UNWIND labels(n) AS label RETURN label, count(*) AS n ORDER BY n DESC"):
    print(f"  {r['label']:<22}{r['n']:>6}")
print("Relationships:")
for r in q("MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n ORDER BY n DESC"):
    print(f"  {r['rel']:<22}{r['n']:>6}")

In [ ]:
import pandas as pd

tables = q("""
MATCH (:Schema {name: $schema})-[:HAS_TABLE]->(t:Table)
OPTIONAL MATCH (t)-[:HAS_COLUMN]->(c:Column)
RETURN t.name AS table, count(c) AS columns, t.embedding IS NOT NULL AS embedded
ORDER BY table
""", schema=SF_SCHEMA)
pd.DataFrame(tables)

In [ ]:
checks = []
def check(name, ok, detail=""):
    checks.append(ok)
    print(("✅" if ok else "❌"), name, ("— " + detail) if detail else "")

n_tables = len(tables)
check("Tables loaded", n_tables > 0, f"{n_tables} tables in {SF_SCHEMA}")
check("Columns loaded", all(t["columns"] > 0 for t in tables) and n_tables > 0)

idx = {r["name"]: r["state"] for r in q("SHOW INDEXES YIELD name, state")}
check("Full-text index online", idx.get("table_full_text_index") == "ONLINE")
if USE_EMBEDDINGS:
    check("Table vector index online", idx.get("table_vector_index") == "ONLINE")
    check("Tables have embeddings", all(t["embedded"] for t in tables))

fks = q("MATCH ()-[r:REFERENCES]->() RETURN count(r) AS n")[0]["n"]
print(f"ℹ️  REFERENCES (foreign key) edges: {fks}")
if fks == 0:
    print("   None found. neocarta builds these from declared PRIMARY/FOREIGN KEY constraints; "
          "the Snowflake sample data (TPCH) declares none, so no join edges are expected here. "
          "Your own schemas with informational keys will get them.")

print("\n🎉 Graph looks good." if all(checks) else "\n⚠️  Some checks failed: scroll up, fix, and re-run section 4.")

## 6 · Try a semantic search

This is the same retrieval the MCP server gives an agent: describe what you are looking for in plain words and get back the matching tables with their columns. (Needs embeddings; otherwise we fall back to full-text search.)

In [ ]:
QUESTION = "customer orders and their prices"

if USE_EMBEDDINGS:
    cmd = ["tool", "get-context-by-table-vector-search", "--text-content", QUESTION, "--max-tables", "3", "--json"]
else:
    cmd = ["tool", "get-context-by-table-full-text-search", "--text-content", QUESTION, "--json"]

rc, out = run_neocarta(*cmd, show=False)
if rc != 0:
    print("❌ Search failed:\n", out[-800:])
else:
    start = out.find("{") if out.find("{") != -1 else out.find("[")
    try:
        data = json.loads(out[start:])
        print(json.dumps(data, indent=2)[:3500])
    except json.JSONDecodeError:
        print(out[-3500:])

## Summary and next steps

You now have a metadata graph of your Snowflake schema. Next:

- **Chat with it:** `uv run workshop-agent` (see `docs/attendee_guide.md` section 7.5): an agent that uses these graph tools to write and run Snowflake SQL.
- **Serve it to your own assistant:** `uv run neocarta-mcp` (section 7.4).
- **Other schemas:** change `SNOWFLAKE_DATABASE` / `SNOWFLAKE_SCHEMA` in `.env` and re-run from section 1.

In [ ]:
driver.close()